# DistilBERT Aspect-Conditioned Sentiment Fine-Tuning with Boosted Neutral Class Weighting
### Google Colab Ready Notebook: 4 Epochs on `modified_annotations_2000_corrected.csv` + Evaluation on Gold 650 Benchmark

This notebook is configured to run end-to-end in **Google Colab** (with GPU acceleration) or locally in VS Code / Jupyter:
1. **Dataset**: Trains on `modified_annotations_2000_corrected.csv` (2,000 food reviews with corrected aspect/sentiment annotations).
2. **Neutral Class Weight Boosting**: In restaurant review datasets, the **Neutral class** is heavily imbalanced (~2.6% of annotations). Standard cross-entropy loss causes the model to ignore neutral nuances. We compute balanced weights and apply an explicit **Neutral Boost Factor** in `nn.CrossEntropyLoss(weight=...)` so the model heavily penalizes neutral misclassifications.
3. **Training Budget**: Exactly **4 Epochs** using `distilbert-base-uncased` with layer freezing, AdamW optimizer, and validation tracking.
4. **Out-of-Distribution Benchmark**: Direct evaluation of the fine-tuned DistilBERT model on the **Gold 650 benchmark dataset** (`gold_aspect_annotated_650_candidate_updated.csv`), reporting **Accuracy**, **Macro-F1**, and **Weighted-F1**.
5. **Colab Export**: One-click checkpoint zipping and browser download.


In [ ]:
# 1. Environment Detection, GPU Verification & Dependencies
import sys

IS_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IS_COLAB}")

if IS_COLAB:
    print("Installing required libraries in Colab environment...")
    !pip install -q transformers torch scikit-learn seaborn matplotlib pandas numpy

import os
import gc
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU not detected. Training will run on CPU. On Colab, go to Runtime -> Change runtime type -> T4 GPU.")

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [ ]:
# 2. File Resolution (Auto-Detects Local Repo, Colab /content, or Upload Widget)

# Search candidates for corrected annotations
TRAIN_CANDIDATES = [
    'modified_annotations_2000_corrected.csv',
    'notebooks/modified_annotations_2000_corrected.csv',
    'final_aspect_evaluation/modified_annotations_2000_corrected.csv',
    '../modified_annotations_2000_corrected.csv',
    '/content/modified_annotations_2000_corrected.csv',
    'final_aspect_evaluation/modified_annotations_2000.csv'
]

GOLD_CANDIDATES = [
    'data/gold/gold_aspect_annotated_650_candidate_updated.csv',
    'gold_aspect_annotated_650_candidate_updated.csv',
    '../data/gold/gold_aspect_annotated_650_candidate_updated.csv',
    '/content/gold_aspect_annotated_650_candidate_updated.csv',
    'data/gold/gold_aspect_annotated_650_candidate.csv',
    'gold_aspect_annotated_650_candidate.csv'
]

TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if os.path.exists(p)), None)
GOLD_PATH = next((p for p in GOLD_CANDIDATES if os.path.exists(p)), None)

# If running on Colab and files are not found, trigger file uploader
if IS_COLAB and (TRAIN_PATH is None or GOLD_PATH is None):
    from google.colab import files
    print("="*70)
    print("Files not found in /content. Please upload:")
    print("1. modified_annotations_2000_corrected.csv")
    print("2. gold_aspect_annotated_650_candidate_updated.csv")
    print("="*70)
    uploaded = files.upload()
    TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if os.path.exists(p)), None)
    GOLD_PATH = next((p for p in GOLD_CANDIDATES if os.path.exists(p)), None)

print(f"Training Dataset: {TRAIN_PATH}")
print(f"Gold Benchmark:   {GOLD_PATH}")

assert TRAIN_PATH is not None, "Training file modified_annotations_2000_corrected.csv not found!"
assert os.path.exists(TRAIN_PATH), f"File does not exist: {TRAIN_PATH}"

In [ ]:
# 3. Data Ingestion, Filtering & Aspect-Conditioned Representation
df_raw = pd.read_csv(TRAIN_PATH)

VALID_ASPECTS = ['Food', 'Service', 'Price / Value', 'Ambience', 'General Experience']
VALID_SENTIMENTS = ['Positive', 'Negative', 'Neutral']
LABEL_MAP = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
INV_LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

mask_eligible = df_raw['aspect'].isin(VALID_ASPECTS) & df_raw['sentiment'].isin(VALID_SENTIMENTS)
df_eligible = df_raw[mask_eligible].copy()

# Construct Aspect-Conditioned input representation
df_eligible['aspect_conditioned_text'] = 'Aspect: ' + df_eligible['aspect'].astype(str) + ' [SEP] ' + df_eligible['clause_text'].astype(str)
df_eligible['label'] = df_eligible['sentiment'].map(LABEL_MAP)

print(f"Total Raw Rows:        {len(df_raw):,}")
print(f"Total Eligible Rows:   {len(df_eligible):,}")
print(f"Unique Reviews:        {df_eligible['review_id'].nunique():,}")
print(f"Unique Clauses:        {df_eligible['clause_id'].nunique():,}")

sent_counts = df_eligible['sentiment'].value_counts()
print("\n--- Sentiment Distribution (Notice extreme Neutral imbalance) ---")
for sent, count in sent_counts.items():
    print(f"  {sent:10s}: {count:,} ({count/len(df_eligible)*100:.2f}%)")

In [ ]:
# 4. Grouped Stratified Partitioning (70% Train / 15% Val / 15% Test)
# Grouped strictly by review_id (0 reviewer leakage across splits)

review_sent = df_eligible.groupby('review_id')['sentiment'].apply(lambda s: s.value_counts().index[0])
reviews_by_strat = {}
for rid, s in review_sent.items():
    reviews_by_strat.setdefault(s, []).append(rid)

np.random.seed(42)
train_r, val_r, test_r = [], [], []

for s, rids in reviews_by_strat.items():
    shuf = np.random.permutation(rids)
    n = len(shuf)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_r.extend(shuf[:n_train])
    val_r.extend(shuf[n_train:n_train + n_val])
    test_r.extend(shuf[n_train + n_val:])

train_df = df_eligible[df_eligible['review_id'].isin(train_r)].copy()
val_df = df_eligible[df_eligible['review_id'].isin(val_r)].copy()
test_df = df_eligible[df_eligible['review_id'].isin(test_r)].copy()

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# Leakage checks
assert len(set(train_r).intersection(set(val_r))) == 0, "Leakage train-val"
assert len(set(train_r).intersection(set(test_r))) == 0, "Leakage train-test"
assert len(set(val_r).intersection(set(test_r))) == 0, "Leakage val-test"

print("Grouped Partition Summary (0 Reviewer Leakage):")
print(f"  Train Set:      {len(train_df):,} rows ({len(train_r)} reviews, {len(train_df)/len(df_eligible)*100:.1f}%)")
print(f"  Validation Set: {len(val_df):,} rows ({len(val_r)} reviews, {len(val_df)/len(df_eligible)*100:.1f}%)")
print(f"  Test Set:       {len(test_df):,} rows ({len(test_r)} reviews, {len(test_df)/len(df_eligible)*100:.1f}%)")
print(f"Train Sentiment Distribution: {train_df['sentiment'].value_counts().to_dict()}")

In [ ]:
# 5. DistilBERT Tokenization & PyTorch DataLoaders
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 64
BATCH_SIZE = 32

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class AspectSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = AspectSentimentDataset(train_df['aspect_conditioned_text'].tolist(), y_train, tokenizer, MAX_LENGTH)
val_dataset = AspectSentimentDataset(val_df['aspect_conditioned_text'].tolist(), y_val, tokenizer, MAX_LENGTH)
test_dataset = AspectSentimentDataset(test_df['aspect_conditioned_text'].tolist(), y_test, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"DataLoaders prepared: {len(train_loader)} train batches ({BATCH_SIZE} per batch).")

In [ ]:
# 6. Boosted Neutral Class Weighting & Model Initialization

# Step A: Compute standard balanced class weights from training distribution
base_weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y_train)

# Step B: Apply Neutral Class Boost Factor
# Neutral represents only ~2.6% of data. Boosting neutral loss penalizes false negatives on Neutral.
NEUTRAL_BOOST_FACTOR = 2.0  # <-- TUNE HERE: 1.5 - 2.5 recommended

class_weights = base_weights.copy()
class_weights[1] = class_weights[1] * NEUTRAL_BOOST_FACTOR  # Multiply Neutral weight

print("="*65)
print("CLASS WEIGHT CONFIGURATION FOR CROSS-ENTROPY LOSS")
print("="*65)
print(f"Base Balanced Weights: [Neg: {base_weights[0]:.3f}, Neu: {base_weights[1]:.3f}, Pos: {base_weights[2]:.3f}]")
print(f"Neutral Boost Factor:  {NEUTRAL_BOOST_FACTOR}x")
print(f"Final Loss Weights:    [Neg: {class_weights[0]:.3f}, Neu: {class_weights[1]:.3f}, Pos: {class_weights[2]:.3f}]")

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Step C: Model Initialization with layer freezing for stability
print(f"\nLoading {MODEL_NAME} for 3-class sequence classification...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

# Freeze first 3 transformer layers for faster training and feature retention
for param in model.distilbert.transformer.layer[:3].parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.1f}%)")

LEARNING_RATE = 3e-5
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=0.01)

In [ ]:
# 7. Fine-Tuning DistilBERT (4 Epochs on GPU)
NUM_EPOCHS = 4
best_val_macro_f1 = -1.0
best_state_dict = None
training_history = []

print("="*70)
print(f"STARTING DISTILBERT FINE-TUNING FOR {NUM_EPOCHS} EPOCHS (DEVICE: {device})")
print("="*70)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Validation Evaluation
    model.eval()
    val_preds = []
    total_val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            v_loss = criterion(outputs.logits, labels)
            total_val_loss += v_loss.item()
            
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = accuracy_score(y_val, val_preds)
    val_macro_f1 = f1_score(y_val, val_preds, average='macro')
    val_weighted_f1 = f1_score(y_val, val_preds, average='weighted')
    val_class_f1 = f1_score(y_val, val_preds, average=None)
    
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_acc': val_acc,
        'val_macro_f1': val_macro_f1,
        'val_weighted_f1': val_weighted_f1,
        'neg_f1': val_class_f1[0],
        'neu_f1': val_class_f1[1],
        'pos_f1': val_class_f1[2]
    })
    
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc*100:.2f}% | Macro-F1: {val_macro_f1*100:.2f}% | Neu-F1: {val_class_f1[1]*100:.2f}%")
          
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  --> New best model checkpoint saved (Macro-F1: {val_macro_f1*100:.2f}%)")

print("\nFine-tuning completed successfully!")

In [ ]:
# 8. Checkpoint Persistence, GPU Garbage Collection & Model Recall
SAVE_DIR = 'outputs/distilbert_neutral_boosted'
os.makedirs(SAVE_DIR, exist_ok=True)

# Load best state dict and save
model.load_state_dict(best_state_dict)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save training history
with open(os.path.join(SAVE_DIR, 'training_history.json'), 'w') as f:
    json.dump(training_history, f, indent=2)

print(f"Best DistilBERT model & tokenizer persisted to: {SAVE_DIR}")

# Delete in-memory training model to clean VRAM
del model, optimizer, best_state_dict
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("In-memory training objects deleted and GPU cache cleared.")
print("Reloading fine-tuned DistilBERT checkpoint from disk onto active device...")

recalled_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR, num_labels=3)
recalled_model.to(device)
recalled_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

print(f"DistilBERT checkpoint recalled onto {device} successfully!")

In [ ]:
# 9. Evaluation on Held-Out Test Set (Internal Split)
recalled_model.eval()
test_preds = []
test_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        logits = recalled_model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        
        test_probs.extend(probs)
        test_preds.extend(preds)

test_preds = np.array(test_preds)
target_names = ['Negative', 'Neutral', 'Positive']

t_acc = accuracy_score(y_test, test_preds)
t_macro_f1 = f1_score(y_test, test_preds, average='macro')
t_weighted_f1 = f1_score(y_test, test_preds, average='weighted')
t_class_f1 = f1_score(y_test, test_preds, average=None)

print("="*75)
print("INTERNAL TEST SPLIT EVALUATION (DISTILBERT NEUTRAL-BOOSTED)")
print("="*75)
print(classification_report(y_test, test_preds, target_names=target_names, digits=4))
print(f"Accuracy:    {t_acc*100:.2f}%")
print(f"Macro-F1:    {t_macro_f1*100:.2f}%")
print(f"Weighted-F1: {t_weighted_f1*100:.2f}%")
print(f"Neutral-F1:  {t_class_f1[1]*100:.2f}%")

# Normalized Confusion Matrix
cm = confusion_matrix(y_test, test_preds, labels=[0, 1, 2])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=target_names, yticklabels=target_names, vmin=0, vmax=1)
ax.set_title(f"Test Split Normalized Confusion Matrix\nMacro-F1: {t_macro_f1*100:.2f}% | Weighted-F1: {t_weighted_f1*100:.2f}%", fontweight='bold')
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 10. Out-of-Distribution Evaluation on Gold Benchmark Sets
# Evaluates on:
# 1. Pristine 30% Untouched Holdout Test Set (N=195, Zero Overlap with 70% Training Mix)
# 2. Full Gold 650 Dataset (N=650)

def evaluate_on_benchmark(csv_path, benchmark_name):
    if not csv_path or not os.path.exists(csv_path):
        print(f"{benchmark_name} dataset not found at: {csv_path}")
        return None
        
    print("="*75)
    print(f"EVALUATION ON {benchmark_name.upper()} ({csv_path})")
    print("="*75)
    
    b_df = pd.read_csv(csv_path)
    aspect_map = {
        'Food': 'Food',
        'Service': 'Service',
        'Ambience': 'Ambience',
        'Price': 'Price / Value',
        'Price / Value': 'Price / Value',
        'General': 'General Experience',
        'General Experience': 'General Experience'
    }
    b_df['canonical_aspect'] = b_df['gold_aspect'].map(lambda a: aspect_map.get(a, 'General Experience'))
    b_df['aspect_conditioned_text'] = 'Aspect: ' + b_df['canonical_aspect'] + ' [SEP] ' + b_df['clause']
    b_df['label'] = b_df['gold_sentiment'].map(LABEL_MAP)
    
    b_dataset = AspectSentimentDataset(
        b_df['aspect_conditioned_text'].tolist(),
        b_df['label'].values,
        recalled_tokenizer,
        MAX_LENGTH
    )
    b_loader = DataLoader(b_dataset, batch_size=64, shuffle=False)
    
    recalled_model.eval()
    preds_list = []
    probs_list = []
    
    with torch.no_grad():
        for batch in b_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = recalled_model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs_list.extend(probs)
            preds_list.extend(np.argmax(probs, axis=1))
            
    preds_arr = np.array(preds_list)
    y_true = b_df['label'].values
    
    acc = accuracy_score(y_true, preds_arr)
    macro_f1 = f1_score(y_true, preds_arr, average='macro')
    weighted_f1 = f1_score(y_true, preds_arr, average='weighted')
    class_f1 = f1_score(y_true, preds_arr, average=None)
    
    print(classification_report(y_true, preds_arr, target_names=target_names, digits=4))
    
    summary = {
        'Benchmark': benchmark_name,
        'N_Samples': len(b_df),
        'Accuracy (%)': round(acc * 100, 2),
        'Macro-F1 (%)': round(macro_f1 * 100, 2),
        'Weighted-F1 (%)': round(weighted_f1 * 100, 2),
        'Negative F1 (%)': round(class_f1[0] * 100, 2),
        'Neutral F1 (%)': round(class_f1[1] * 100, 2),
        'Positive F1 (%)': round(class_f1[2] * 100, 2)
    }
    
    # Normalized Confusion Matrix
    cm = confusion_matrix(y_true, preds_arr, labels=[0, 1, 2])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names, vmin=0, vmax=1)
    ax.set_title(f"{benchmark_name}\nMacro-F1: {macro_f1*100:.2f}% | Weighted-F1: {weighted_f1*100:.2f}%", fontweight='bold')
    ax.set_xlabel('Predicted Label', fontweight='bold')
    ax.set_ylabel('True Label', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return summary

# Benchmark Candidates
HOLD_30_CANDIDATES = [
    'data/gold/gold_650_test_remaining_30pct.csv',
    'gold_650_test_remaining_30pct.csv',
    '../data/gold/gold_650_test_remaining_30pct.csv',
    '/content/gold_650_test_remaining_30pct.csv'
]
HOLD_30_PATH = next((p for p in HOLD_30_CANDIDATES if os.path.exists(p)), None)

all_summaries = []

# 1. Pristine 30% Untouched Holdout Test Set (Zero Overlap with Training)
if HOLD_30_PATH:
    res_30 = evaluate_on_benchmark(HOLD_30_PATH, "Gold 650 Pristine Holdout Test Set (30% Split, N=195)")
    if res_30:
        all_summaries.append(res_30)

# 2. Full Gold 650 Dataset
if GOLD_PATH:
    res_full = evaluate_on_benchmark(GOLD_PATH, "Full Gold 650 Benchmark (N=650)")
    if res_full:
        all_summaries.append(res_full)

if all_summaries:
    print("\n=================== OVERALL BENCHMARK COMPARISON TABLE ===================")
    print(pd.DataFrame(all_summaries).to_string(index=False))

In [ ]:
# 11. Colab Model Export: Create Zip Archive & Download
zip_path = 'distilbert_neutral_boosted.zip'
print(f"Compressing {SAVE_DIR} to {zip_path}...")
shutil.make_archive('distilbert_neutral_boosted', 'zip', SAVE_DIR)
print(f"Archive created: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

if IS_COLAB:
    from google.colab import files
    print("Initiating browser download of trained model weights...")
    files.download(zip_path)
else:
    print(f"Model checkpoint zip ready at: {os.path.abspath(zip_path)}")